# CEHARPS 04 — ฝึก U-Net และ DeepLabV3+ / Train U-Net and DeepLabV3+

ใช้ข้อมูล MagSet-2 เฉพาะการจำแนกถิ่นที่อยู่ป่าชายเลน ไม่ใช้เป็นป้าย MHI และใช้ AMP API รุ่นใหม่ของ PyTorch


In [ ]:
import json  # TH: นำเข้าเครื่องมือ JSON | EN: Import JSON utilities.
import subprocess  # TH: นำเข้าเครื่องมือเรียกคำสั่งระบบ | EN: Import subprocess utilities.
import sys  # TH: นำเข้าข้อมูลตัวแปลภาษา Python | EN: Import Python runtime information.
from pathlib import Path  # TH: นำเข้าคลาสจัดการพาธ | EN: Import the path-management class.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8", "rasterio>=1.3,<2"])  # TH: ติดตั้งไลบรารีโมเดลภาพที่กำหนดรุ่น | EN: Install pinned image-model libraries.
import albumentations as A  # TH: นำเข้าเครื่องมือเพิ่มความหลากหลายภาพ | EN: Import image augmentation tools.
import numpy as np  # TH: นำเข้า NumPy | EN: Import NumPy.
import pandas as pd  # TH: นำเข้า pandas | EN: Import pandas.
import rasterio  # TH: นำเข้าเครื่องมืออ่าน GeoTIFF | EN: Import GeoTIFF reading utilities.
import segmentation_models_pytorch as smp  # TH: นำเข้า U-Net และ DeepLabV3+ | EN: Import U-Net and DeepLabV3+ implementations.
import torch  # TH: นำเข้า PyTorch | EN: Import PyTorch.
from albumentations.pytorch import ToTensorV2  # TH: นำเข้าตัวแปลงภาพเป็น tensor | EN: Import image-to-tensor conversion.
from google.colab import drive  # TH: นำเข้าเครื่องมือเชื่อม Drive | EN: Import the Drive connector.
from torch.utils.data import DataLoader, Dataset  # TH: นำเข้าโครงสร้างชุดข้อมูลและตัวโหลด | EN: Import dataset and loader structures.
drive.mount("/content/drive")  # TH: เชื่อม Google Drive | EN: Mount Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/CEHARPS")  # TH: กำหนดโฟลเดอร์โครงการ | EN: Define the project folder.
CONFIG = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))  # TH: อ่านค่ากลาง | EN: Load shared settings.
SEED = int(CONFIG["seed"])  # TH: อ่านค่าเมล็ดสุ่ม | EN: Read the random seed.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # TH: เลือก GPU เมื่อพร้อม | EN: Select a GPU when available.
torch.manual_seed(SEED)  # TH: ตั้งค่าเมล็ดสุ่มของ PyTorch | EN: Seed PyTorch.
if torch.cuda.is_available():  # TH: ตรวจว่ามี CUDA หรือไม่ | EN: Check whether CUDA is available.
    torch.cuda.manual_seed_all(SEED)  # TH: ตั้งค่าเมล็ดสุ่มของ GPU | EN: Seed all CUDA devices.
print("Device:", DEVICE)  # TH: แสดงอุปกรณ์ที่ใช้ฝึก | EN: Display the training device.


In [ ]:
ROOT = PROJECT_ROOT / "data/segmentation/magset2_sample"  # TH: กำหนดโฟลเดอร์ข้อมูลภาพ | EN: Define the segmentation-data folder.
MANIFEST = pd.read_csv(ROOT / "manifest.csv")  # TH: อ่าน manifest ของภาพและหน้ากาก | EN: Load the image-mask manifest.
IMAGE_SIZE = int(CONFIG["image_size"])  # TH: อ่านขนาดภาพเข้าโมเดล | EN: Read the model input size.
BATCH_SIZE = int(CONFIG["batch_size"])  # TH: อ่านขนาดแบตช์ | EN: Read the batch size.
NUM_CLASSES = 2  # TH: กำหนด background และ mangrove | EN: Define background and mangrove classes.
CLASS_NAMES = ["background", "mangrove"]  # TH: กำหนดชื่อคลาส | EN: Define class names.
BANDS = (3, 2, 1)  # TH: อ่าน Red Green Blue จากลำดับ B G R ของชุดตัวอย่าง | EN: Read RGB from the sample's B-G-R ordering.

def scale_percentile(image: np.ndarray) -> np.ndarray:  # TH: สร้างฟังก์ชันปรับช่วงค่าภาพ | EN: Define percentile image scaling.
    output = np.zeros_like(image, dtype=np.float32)  # TH: เตรียมภาพผลลัพธ์ | EN: Initialize the scaled output.
    for channel in range(image.shape[2]):  # TH: วนปรับค่าทีละแชนเนล | EN: Scale each channel separately.
        values = image[..., channel].astype(np.float32)  # TH: อ่านค่าแชนเนลเป็น float | EN: Read the channel as floats.
        low, high = np.percentile(values, [2, 98])  # TH: หาเปอร์เซ็นไทล์ 2 และ 98 | EN: Calculate the 2nd and 98th percentiles.
        output[..., channel] = 0.0 if high <= low else np.clip((values - low) / (high - low), 0, 1)  # TH: ปรับค่าให้อยู่ 0–1 อย่างปลอดภัย | EN: Safely scale values to 0–1.
    return output  # TH: คืนภาพที่ปรับช่วงแล้ว | EN: Return the scaled image.

def read_image(path: str) -> np.ndarray:  # TH: สร้างฟังก์ชันอ่านภาพหลายแบนด์ | EN: Define a multiband image reader.
    with rasterio.open(path) as source:  # TH: เปิด GeoTIFF | EN: Open the GeoTIFF.
        image = source.read(BANDS).transpose(1, 2, 0)  # TH: อ่าน RGB และย้ายแชนเนลไปแกนสุดท้าย | EN: Read RGB and move channels last.
    return scale_percentile(image)  # TH: คืนภาพที่ปรับช่วงแล้ว | EN: Return the scaled image.

def read_mask(path: str) -> np.ndarray:  # TH: สร้างฟังก์ชันอ่านหน้ากาก | EN: Define a mask reader.
    mask = np.load(path, allow_pickle=False).astype(np.int64)  # TH: อ่านหน้ากากจำนวนเต็มโดยไม่ใช้ pickle | EN: Load the integer mask without pickle.
    if mask.min() < 0 or mask.max() >= NUM_CLASSES:  # TH: ตรวจช่วงรหัสคลาส | EN: Validate mask class IDs.
        raise ValueError(f"Invalid mask labels in {path}")  # TH: หยุดเมื่อรหัสคลาสผิด | EN: Stop on invalid class IDs.
    return mask  # TH: คืนหน้ากากที่ตรวจแล้ว | EN: Return the validated mask.


In [ ]:
def transform(training: bool) -> A.Compose:  # TH: สร้างฟังก์ชันกำหนดการแปลงภาพ | EN: Define the image-transform builder.
    steps = [A.Resize(IMAGE_SIZE, IMAGE_SIZE)]  # TH: เริ่มด้วยการปรับขนาดภาพและหน้ากากพร้อมกัน | EN: Start by resizing image and mask together.
    if training:  # TH: เพิ่ม augmentation เฉพาะชุดฝึก | EN: Add augmentation only for training.
        steps += [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5)]  # TH: เพิ่มการกลับและหมุนภาพ | EN: Add flips and right-angle rotations.
    steps += [A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=1.0), ToTensorV2()]  # TH: ทำ normalization และแปลงเป็น tensor | EN: Normalize and convert to tensors.
    return A.Compose(steps)  # TH: รวมขั้นตอนเป็นชุดเดียว | EN: Compose the transform sequence.

class HabitatDataset(Dataset):  # TH: ประกาศชุดข้อมูลสำหรับ segmentation | EN: Define the segmentation dataset.
    def __init__(self, rows: pd.DataFrame, training: bool):  # TH: รับตารางข้อมูลและสถานะการฝึก | EN: Accept rows and training mode.
        self.rows = rows.reset_index(drop=True)  # TH: รีเซ็ตดัชนีตาราง | EN: Reset table indices.
        self.transform = transform(training)  # TH: สร้างการแปลงตามโหมด | EN: Build transforms for the mode.
    def __len__(self) -> int:  # TH: กำหนดวิธีนับจำนวนตัวอย่าง | EN: Define dataset length.
        return len(self.rows)  # TH: คืนจำนวนแถว | EN: Return the row count.
    def __getitem__(self, index: int):  # TH: กำหนดวิธีอ่านตัวอย่างหนึ่งรายการ | EN: Define single-sample loading.
        row = self.rows.iloc[index]  # TH: อ่านแถวตามดัชนี | EN: Read the indexed row.
        result = self.transform(image=read_image(row["image"]), mask=read_mask(row["mask"]))  # TH: อ่านและแปลงภาพกับหน้ากากแบบสอดคล้องกัน | EN: Read and jointly transform image and mask.
        return result["image"].float(), result["mask"].long()  # TH: คืน tensor ภาพและคลาส | EN: Return image and class tensors.

datasets = {name: HabitatDataset(MANIFEST.loc[MANIFEST["split"] == name], training=name == "train") for name in ["train", "val", "test"]}  # TH: สร้างชุดข้อมูลสามส่วน | EN: Build train, validation, and test datasets.
loaders = {name: DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=name == "train", num_workers=int(CONFIG["num_workers"]), pin_memory=DEVICE.type == "cuda") for name, dataset in datasets.items()}  # TH: สร้างตัวโหลดของแต่ละชุด | EN: Build a loader for each split.
for name, dataset in datasets.items():  # TH: วนตรวจจำนวนตัวอย่าง | EN: Iterate through dataset sizes.
    if len(dataset) == 0:  # TH: ตรวจว่าชุดข้อมูลว่างหรือไม่ | EN: Check whether a split is empty.
        raise ValueError(f"Empty segmentation split: {name}")  # TH: หยุดเมื่อพบชุดว่าง | EN: Stop on an empty split.
    print(name, len(dataset))  # TH: แสดงจำนวนตัวอย่าง | EN: Display the split size.


In [ ]:
def build_model(name: str) -> torch.nn.Module:  # TH: สร้างฟังก์ชันเลือกสถาปัตยกรรม | EN: Define the architecture builder.
    common = {"encoder_name": str(CONFIG["encoder_name"]), "encoder_weights": CONFIG["encoder_weights"], "in_channels": 3, "classes": NUM_CLASSES}  # TH: กำหนดค่าที่ใช้ร่วมกัน | EN: Define shared model settings.
    if name == "unet":  # TH: ตรวจว่าต้องการ U-Net หรือไม่ | EN: Check whether U-Net is requested.
        return smp.Unet(**common)  # TH: สร้าง U-Net | EN: Create U-Net.
    if name == "deeplabv3plus":  # TH: ตรวจว่าต้องการ DeepLabV3+ หรือไม่ | EN: Check whether DeepLabV3+ is requested.
        return smp.DeepLabV3Plus(**common)  # TH: สร้าง DeepLabV3+ | EN: Create DeepLabV3+.
    raise ValueError(f"Unsupported model: {name}")  # TH: หยุดเมื่อชื่อโมเดลไม่รองรับ | EN: Stop on an unsupported model name.

dice_loss = smp.losses.DiceLoss(mode="multiclass", from_logits=True)  # TH: สร้าง Dice loss สำหรับหลายคลาส | EN: Create multiclass Dice loss.
ce_loss = torch.nn.CrossEntropyLoss()  # TH: สร้าง Cross-Entropy loss | EN: Create cross-entropy loss.

def loss_function(logits, masks):  # TH: สร้างฟังก์ชัน loss รวม | EN: Define the combined loss.
    return ce_loss(logits, masks) + dice_loss(logits, masks)  # TH: รวม Cross-Entropy กับ Dice | EN: Combine cross-entropy and Dice losses.

def metrics_from_confusion(confusion: torch.Tensor) -> dict:  # TH: สร้างฟังก์ชันคำนวณ mIoU และ Dice | EN: Define mIoU and Dice calculation.
    matrix = confusion.float()  # TH: แปลง confusion matrix เป็น float | EN: Convert the confusion matrix to floats.
    true_positive = torch.diag(matrix)  # TH: อ่านค่าทำนายถูกแต่ละคลาส | EN: Read per-class true positives.
    union = matrix.sum(0) + matrix.sum(1) - true_positive  # TH: คำนวณ union ของแต่ละคลาส | EN: Calculate per-class union.
    dice_denominator = matrix.sum(0) + matrix.sum(1)  # TH: คำนวณตัวส่วน Dice | EN: Calculate the Dice denominator.
    iou = torch.where(union > 0, true_positive / union.clamp_min(1), torch.nan)  # TH: คำนวณ IoU อย่างปลอดภัย | EN: Safely calculate IoU.
    dice = torch.where(dice_denominator > 0, 2 * true_positive / dice_denominator.clamp_min(1), torch.nan)  # TH: คำนวณ Dice อย่างปลอดภัย | EN: Safely calculate Dice.
    return {"mIoU": float(torch.nanmean(iou)), "mean_dice": float(torch.nanmean(dice)), "per_class_iou": {CLASS_NAMES[i]: None if torch.isnan(iou[i]) else float(iou[i]) for i in range(NUM_CLASSES)}}  # TH: คืนตัวชี้วัดพร้อมรายคลาส | EN: Return aggregate and per-class metrics.


In [ ]:
def run_epoch(model, loader, optimizer=None) -> dict:  # TH: สร้างฟังก์ชันรันหนึ่ง epoch | EN: Define a one-epoch runner.
    training = optimizer is not None  # TH: ตรวจว่าเป็นโหมดฝึกหรือประเมิน | EN: Detect training or evaluation mode.
    model.train(training)  # TH: ตั้งสถานะโมเดลให้ตรงกับโหมด | EN: Set the model mode.
    confusion = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64)  # TH: เตรียม confusion matrix | EN: Initialize the confusion matrix.
    total_loss = 0.0  # TH: เตรียมผลรวม loss | EN: Initialize total loss.
    amp_enabled = DEVICE.type == "cuda"  # TH: เปิด AMP เมื่อใช้ CUDA | EN: Enable AMP on CUDA.
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)  # TH: สร้างตัวปรับสเกล gradient ด้วย API ใหม่ | EN: Create a gradient scaler with the current API.
    for images, masks in loader:  # TH: วนอ่านแต่ละแบตช์ | EN: Iterate through batches.
        images, masks = images.to(DEVICE), masks.to(DEVICE)  # TH: ย้ายข้อมูลไปอุปกรณ์ฝึก | EN: Move data to the training device.
        if training:  # TH: ล้าง gradient เฉพาะโหมดฝึก | EN: Clear gradients only during training.
            optimizer.zero_grad(set_to_none=True)  # TH: ล้าง gradient อย่างมีประสิทธิภาพ | EN: Efficiently clear gradients.
        with torch.set_grad_enabled(training):  # TH: เปิด gradient เฉพาะโหมดฝึก | EN: Enable gradients only for training.
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp_enabled):  # TH: ใช้ mixed precision เมื่อเหมาะสม | EN: Use mixed precision when appropriate.
                logits = model(images)  # TH: คำนวณผลลัพธ์โมเดล | EN: Run the model forward pass.
                loss = loss_function(logits, masks)  # TH: คำนวณ loss | EN: Calculate the loss.
            if training:  # TH: ทำ backpropagation เฉพาะโหมดฝึก | EN: Backpropagate only during training.
                scaler.scale(loss).backward()  # TH: คำนวณ gradient ที่ปรับสเกล | EN: Backpropagate the scaled loss.
                scaler.step(optimizer)  # TH: ปรับน้ำหนักโมเดล | EN: Update model weights.
                scaler.update()  # TH: ปรับค่าตัวคูณสเกล | EN: Update the gradient scale.
        predictions = logits.argmax(dim=1).detach().cpu()  # TH: เลือกคลาสที่คะแนนสูงสุด | EN: Select the highest-scoring class.
        targets = masks.detach().cpu()  # TH: ย้ายหน้ากากจริงกลับ CPU | EN: Move targets back to CPU.
        valid = (targets >= 0) & (targets < NUM_CLASSES)  # TH: เลือกพิกเซลที่รหัสคลาสถูกต้อง | EN: Select pixels with valid class IDs.
        indices = NUM_CLASSES * targets[valid] + predictions[valid]  # TH: แปลงคู่จริง–ทำนายเป็นดัชนี | EN: Encode target-prediction pairs as indices.
        confusion += torch.bincount(indices, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)  # TH: สะสม confusion matrix | EN: Accumulate the confusion matrix.
        total_loss += float(loss.item()) * images.size(0)  # TH: สะสม loss ตามจำนวนภาพ | EN: Accumulate sample-weighted loss.
    result = metrics_from_confusion(confusion)  # TH: คำนวณตัวชี้วัดจาก confusion matrix | EN: Calculate metrics from the confusion matrix.
    result["loss"] = total_loss / len(loader.dataset)  # TH: เพิ่มค่า loss เฉลี่ย | EN: Add the average loss.
    return result  # TH: คืนผล epoch | EN: Return epoch results.


In [ ]:
ARTIFACTS = PROJECT_ROOT / "artifacts/segmentation"  # TH: กำหนดโฟลเดอร์ผลโมเดลภาพ | EN: Define the segmentation artifact folder.
ARTIFACTS.mkdir(parents=True, exist_ok=True)  # TH: สร้างโฟลเดอร์ผลลัพธ์ | EN: Create the artifact folder.

def train_model(name: str) -> dict:  # TH: สร้างฟังก์ชันฝึกโมเดลหนึ่งชนิด | EN: Define single-model training.
    model = build_model(name).to(DEVICE)  # TH: สร้างและย้ายโมเดลไปอุปกรณ์ | EN: Build and move the model to the device.
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(CONFIG["learning_rate"]), weight_decay=0.0001)  # TH: สร้าง AdamW optimizer | EN: Create the AdamW optimizer.
    best_iou, stale = -1.0, 0  # TH: เริ่มค่าผลงานดีที่สุดและรอบที่ไม่ดีขึ้น | EN: Initialize best score and stale-epoch count.
    checkpoint = ARTIFACTS / f"{name}_best.pt"  # TH: กำหนดไฟล์ checkpoint | EN: Define the checkpoint path.
    history = []  # TH: เตรียมรายการประวัติการฝึก | EN: Initialize training history.
    for epoch in range(1, int(CONFIG["segmentation_epochs"]) + 1):  # TH: วนตามจำนวน epoch ที่กำหนด | EN: Iterate through configured epochs.
        train_result = run_epoch(model, loaders["train"], optimizer)  # TH: ฝึกหนึ่ง epoch | EN: Train for one epoch.
        val_result = run_epoch(model, loaders["val"])  # TH: ประเมิน validation | EN: Evaluate validation performance.
        row = {"epoch": epoch, "train_loss": train_result["loss"], "train_mIoU": train_result["mIoU"], "val_loss": val_result["loss"], "val_mIoU": val_result["mIoU"], "val_mean_dice": val_result["mean_dice"]}  # TH: สร้างแถวประวัติการฝึก | EN: Build a training-history row.
        history.append(row)  # TH: เพิ่มประวัติรอบปัจจุบัน | EN: Append the current epoch record.
        print(name, row)  # TH: แสดงผลรอบปัจจุบัน | EN: Display the current epoch result.
        if val_result["mIoU"] > best_iou:  # TH: ตรวจว่าผล validation ดีขึ้นหรือไม่ | EN: Check for validation improvement.
            best_iou, stale = val_result["mIoU"], 0  # TH: อัปเดตคะแนนดีที่สุด | EN: Update the best score.
            torch.save({"state_dict": model.state_dict(), "model_name": name, "best_val_mIoU": best_iou, "class_names": CLASS_NAMES, "image_size": IMAGE_SIZE}, checkpoint)  # TH: บันทึก checkpoint ที่ดีที่สุด | EN: Save the best checkpoint.
        else:  # TH: จัดการกรณีผลไม่ดีขึ้น | EN: Handle no improvement.
            stale += 1  # TH: เพิ่มจำนวนรอบที่ไม่ดีขึ้น | EN: Increment stale epochs.
            if stale >= int(CONFIG["early_stopping_patience"]):  # TH: ตรวจเงื่อนไขหยุดเร็ว | EN: Check the early-stopping condition.
                break  # TH: ออกจากลูปฝึก | EN: Stop training.
    pd.DataFrame(history).to_csv(ARTIFACTS / f"{name}_history.csv", index=False)  # TH: บันทึกประวัติการฝึก | EN: Save training history.
    saved = torch.load(checkpoint, map_location=DEVICE, weights_only=True)  # TH: โหลด checkpoint ที่ดีที่สุดอย่างปลอดภัย | EN: Safely load the best checkpoint.
    model.load_state_dict(saved["state_dict"])  # TH: นำน้ำหนักดีที่สุดกลับเข้าโมเดล | EN: Restore the best weights.
    test_result = run_epoch(model, loaders["test"])  # TH: ประเมินชุด test ครั้งสุดท้าย | EN: Perform final test evaluation.
    (ARTIFACTS / f"{name}_test_metrics.json").write_text(json.dumps(test_result, ensure_ascii=False, indent=2), encoding="utf-8")  # TH: บันทึกตัวชี้วัด test | EN: Save test metrics.
    return test_result  # TH: คืนผลทดสอบ | EN: Return test results.

final_results = {name: train_model(name) for name in ["unet", "deeplabv3plus"]}  # TH: ฝึก U-Net และ DeepLabV3+ แยกกัน | EN: Train U-Net and DeepLabV3+ separately.
print(json.dumps(final_results, ensure_ascii=False, indent=2))  # TH: แสดงผลทดสอบทั้งสองโมเดล | EN: Display both test results.
